# LangChain: Evaluation & Observability

## Outline
* چرا evaluation مهم است؟
* ساخت test cases دستی
* LLM-as-Judge — ارزیابی با LLM
* Tracing با LangSmith
* Debug با `stream_mode`


In [7]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)


## ۱. چرا Evaluation؟

وقتی یه LLM application می‌سازید، باید بدونید:
- آیا جواب‌ها **درست** هستند؟
- آیا بعد از تغییر prompt **بهتر** شدند؟
- کجا **fail** می‌کند؟

روش‌های ارزیابی:
1. **دستی** — نمونه‌های test با جواب صحیح
2. **LLM-as-Judge** — یه LLM دیگه جواب‌ها رو ارزیابی می‌کنه
3. **LangSmith** — ابزار رسمی monitoring و evaluation


## ۲. ساخت Application برای ارزیابی

In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough

# Knowledge base کوچک برای دمو
docs = [
    Document(page_content="ست لباس راحتی Cozy Comfort دارای جیب‌های کناری است و قابلیت شستشو با ماشین لباسشویی را دارد.", metadata={"id": 1}),
    Document(page_content="کاپشن Ultra-Lofty 850 Stretch Down Hooded Jacket متعلق به مجموعه DownTek است.", metadata={"id": 2}),
    Document(page_content="پیراهن Sun Shield دارای استاندارد UPF 50+ بوده و 98 درصد از اشعه فرابنفش را مسدود می‌کند.", metadata={"id": 3}),
    Document(page_content="کفش‌های Hiking Boots ضدآب هستند و از مچ پا پشتیبانی می‌کنند.", metadata={"id": 4}),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "فقط بر اساس متن زمینه زیر پاسخ بده. اگر پاسخ را نمی‌دانی، بگو «نمی‌دانم».\n\nزمینه: {context}"),
    ("human", "{question}"),
])

def format_docs(docs): return "\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

print("RAG chain ready!")

KeyboardInterrupt: 

## ۳. Test Cases دستی

In [ ]:
# test cases با جواب صحیح
test_cases = [
    {
        "question": "آیا ست لباس راحتی Cozy Comfort دارای جیب‌های کناری است؟",
        "expected_answer": "بله"
    },
    {
        "question": "کاپشن Ultra-Lofty 850 Stretch Down Hooded Jacket متعلق به کدام مجموعه است؟",
        "expected_answer": "مجموعه DownTek"
    },
    {
        "question": "درجه UPF پیراهن Sun Shield چقدر است؟",
        "expected_answer": "UPF 50+"
    },
    {
        "question": "آیا کفش‌های Hiking Boots ضدآب هستند؟",
        "expected_answer": "بله"
    },
]
# اجرای RAG chain روی همه test cases
predictions = []
for tc in test_cases:
    predicted = rag_chain.invoke(tc["question"])
    predictions.append({
        "question": tc["question"],
        "expected": tc["expected_answer"],
        "predicted": predicted,
    })
    print(f"Q: {tc['question'][:60]}")
    print(f"Expected: {tc['expected_answer']}")
    print(f"Predicted: {predicted[:100]}")
    print()


## ۴. LLM-as-Judge — ارزیابی با LLM

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class EvalResult(BaseModel):
    """نتیجه ارزیابی یک جواب"""
    grade: Literal["CORRECT", "INCORRECT", "PARTIAL"]
    reasoning: str = Field(description="توضیح چرا این grade داده شد")

eval_llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
eval_structured = eval_llm.with_structured_output(EvalResult)

eval_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert evaluator. 
Grade the predicted answer compared to the expected answer.
- CORRECT: The prediction captures the key information from the expected answer
- PARTIAL: The prediction has some correct information but is incomplete  
- INCORRECT: The prediction is wrong or completely misses the point"""),
    ("human", """Question: {question}
Expected Answer: {expected}
Predicted Answer: {predicted}

Grade this prediction:"""),
])

eval_chain = eval_prompt | eval_structured


In [ ]:
# ارزیابی همه predictions
print("=== Evaluation Results ===\n")
results = []

for pred in predictions:
    result = eval_chain.invoke({
        "question": pred["question"],
        "expected": pred["expected"],
        "predicted": pred["predicted"],
    })
    results.append(result)
    
    print(f"Q: {pred['question'][:60]}")
    print(f"Grade: {result.grade}")
    print(f"Reasoning: {result.reasoning}")
    print()

# خلاصه آماری
from collections import Counter
grade_counts = Counter(r.grade for r in results)
total = len(results)
print("\n=== Summary ===")
for grade, count in grade_counts.items():
    print(f"{grade}: {count}/{total} ({count/total*100:.0f}%)")


## ۵. Tracing با LangSmith

LangSmith ابزار رسمی LangChain برای monitoring و debugging است.

**نصب:**
```bash
pip install langsmith
```

**API Key رایگان:** https://smith.langchain.com


In [ ]:
# فعال‌سازی LangSmith tracing
# در .env فایل اضافه کنید:
# LANGSMITH_TRACING=true
# LANGSMITH_API_KEY=your_key
# LANGSMITH_ENDPOINT=https://api.smith.langchain.com
# LANGSMITH_PROJECT=my-project

# بررسی وضعیت
import os
_ = load_dotenv(find_dotenv())
tracing_enabled = os.environ.get("LANGSMITH_TRACING", "false")
print(f"LangSmith tracing: {tracing_enabled}")
if tracing_enabled == "true":
    print("✓ LangSmith is active.")
    # هر invoke اتوماتیک trace می‌شه
    response = rag_chain.invoke("آیا ست لباس راحتی Cozy Comfort دارای جیب‌های کناری است؟")
    print(f"Response: {response}")
else:
    print("To enable: set LANGSMITH_TRACING=true and LANGSMITH_API_KEY in .env")

## ۶. Debug با stream_mode

In [21]:
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_products(query: str) -> str:
    """Search for products in the catalog."""
    docs = retriever.invoke(query)  # ← اینجا عوض شد
    return "\n".join([doc.page_content for doc in docs])
    
agent = create_agent(
    model=llm,
    tools=[search_products],
    system_prompt="You are a store assistant.",
    checkpointer=InMemorySaver(),
)

# debug: همه steps رو می‌بینید
print("=== Debug Mode (stream_mode='values') ===")
config = {"configurable": {"thread_id": "eval-debug"}}

for step in agent.stream(
    {"messages": [{"role": "user", "content": "چه مصحولاتی برای محافظت در برابر نور خورشید مناسب است؟"}]},
    config=config,
    stream_mode="values"
):
    last_msg = step["messages"][-1]
    msg_type = type(last_msg).__name__
    
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        for tc in last_msg.tool_calls:
            print(f"[{msg_type}] → Calling: {tc['name']}({tc['args']})")
    elif hasattr(last_msg, "name"):
        print(f"[Tool Result] {last_msg.content[:200]}")
    elif last_msg.content:
        print(f"[{msg_type}] {last_msg.content[:400]}")


=== Debug Mode (stream_mode='values') ===
[Tool Result] چه مصحولاتی برای محافظت در برابر نور خورشید مناسب است؟
[AIMessage] → Calling: search_products({'query': 'محافظت در برابر نور خورشید'})
[Tool Result] پیراهن Sun Shield دارای استاندارد UPF 50+ بوده و 98 درصد از اشعه فرابنفش را مسدود می‌کند.
کفش‌های Hiking Boots ضدآب هستند و از مچ پا پشتیبانی می‌کنند.
[Tool Result] برای محافظت در برابر نور خورشید، می‌توانید از محصولات زیر استفاده کنید:

1. **پیراهن Sun Shield**: این پیراهن دارای استاندارد UPF 50+ است و 98 درصد از اشعه فرابنفش را مسدود می‌کند.

2. **کفش‌های Hikin
